# 03 · Shale volume, core-calibrated porosity and permeability
**F-O gas field — Notebook 03** (input: the unzipped **F-O_composites_v3** folder; output: composite **v4**)

What this notebook decides, with numbers you can quote:
1. **Shale volume** from GR (per-well clean/shale endpoints) and, where spectral GR exists, from thorium.
2. **Which density-porosity model fits core best**, tested on ~770 good-hole plugs in 6 wells, with a **leave-one-well-out** blind test.
3. **Permeability**: porosity–permeability transforms per well and field-wide, the porosity cut-off at 0.1 mD, and NMR
   permeability against core in F-O4.

Core helium porosity on cleaned, dried plugs is close to **total** porosity, so the log porosity calibrated here is PHIT.

In [ ]:
# Cell 1 — paths (EDIT)
from pathlib import Path
V3_DIR  = Path(r"C:\Users\YOUR_NAME\...\F-O_composites_v3")   # unzipped F-O_composites_v3 (LAS, CSV, core_data_shifted.csv)
OUT_DIR = Path(r"C:\Users\YOUR_NAME\...\F-O_v4_outputs")
OUT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
# Cell 2 — load and harmonise (unit fixes, RHOB_USE, spectral-GR and NMR-permeability despike)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, lasio, warnings
warnings.filterwarnings("ignore")
from petrotools import petromodels as pm
WELLS = ["F-O1", "F-O2", "F-O3", "F-O4", "F-O6", "F-O8", "F-R1", "F-S1"]
logs = {w: pm.harmonise(w, pd.read_csv(V3_DIR / f"{w}_composite_v3.csv", index_col=0)) for w in WELLS}
core = pd.read_csv(V3_DIR / "core_data_shifted.csv")
CURVES = ["GR", "RHOB_USE", "NPHI", "PEF", "BADHOLE", "THOR", "POTA", "KTIM", "KSDR"]
cl = pm.match_logs(core, logs, CURVES)
ev = cl[cl.phi_pct.notna() & cl.RHOB_USE.notna() & (cl.BADHOLE != 1)].copy()
print(len(core), "plugs;", len(ev), "with good-hole density at log depth")

## Step 1 — shale volume
Clean and shale GR are the 5th and 95th percentiles of GR in each well's reservoir window (±60 m), so the endpoints follow each
well's tool and borehole. Where spectral GR exists (F-O6, F-O8) a thorium-based Vsh is also computed: the Chemostrat **Low-K
unit** at the top of the reservoir is defined by low potassium, so total GR under-reads clay there and thorium is the better
clay indicator. There is no core clay measurement to calibrate against, so Vsh is reported as an **index** for now (XRD from the
F-O6 SCAL can test it later).

In [ ]:
# Cell 3
ep = []
for w in WELLS:
    win = pm.WINDOWS[w]; c, s = pm.gr_endpoints(logs[w], win)
    d = logs[w]; igr = pm.vsh_linear(d.GR, c, s); d["VSH_GR"] = igr; d["VSH_GRL"] = pm.vsh_larionov_old(igr)
    row = dict(well=w, window=f"{win[0]:.0f}-{win[1]:.0f}", GR_clean=round(c), GR_shale=round(s))
    if "THOR" in d and d.THOR.notna().sum() > 100:
        tc, ts = pm.gr_endpoints(d, win, "THOR"); d["VSH_TH"] = pm.vsh_linear(d.THOR, tc, ts); row.update(TH_clean=round(tc, 1), TH_shale=round(ts, 1))
    ep.append(row)
endpoints = pd.DataFrame(ep); endpoints.to_csv(OUT_DIR / "vsh_endpoints.csv", index=False); endpoints

## Step 2 — which porosity model matches core?

In [ ]:
# Cell 4 — five candidate models, scored against core per well
rhog = ev.groupby("well").rhog.median(); ev["rg"] = ev.well.map(rhog)
ev["A_2.65"] = 100 * (2.65 - ev.RHOB_USE) / 1.65
ev["B_coreRhoMa"] = 100 * (ev.rg - ev.RHOB_USE) / (ev.rg - 1.0)
bias_B = (ev.B_coreRhoMa - ev.phi_pct).groupby(ev.well).mean()
ev["C_B+calib"] = ev.B_coreRhoMa - ev.well.map(bias_B)
ev["D_ND_gas"] = np.sqrt((ev.B_coreRhoMa.clip(lower=0) ** 2 + (100 * (ev.NPHI + 0.04)).clip(lower=0) ** 2) / 2)
vsh = ev.apply(lambda r: logs[r.well].VSH_GR.iloc[int(np.searchsorted(logs[r.well].index, r.depth_log_m))], axis=1)
ev["E_B-shale"] = ev.B_coreRhoMa - vsh * 15
rows = []
for m in ["A_2.65", "B_coreRhoMa", "C_B+calib", "D_ND_gas", "E_B-shale"]:
    for w, g in list(ev.groupby("well")) + [("ALL", ev)]:
        rows.append(dict(model=m, well=w, **pm.error_stats(g[m], g.phi_pct)))
score = pd.DataFrame(rows); score.to_csv(OUT_DIR / "porosity_model_scores.csv", index=False)
score.pivot(index="well", columns="model", values="rmse")

In [ ]:
score.pivot(index="well", columns="model", values="bias")

**Decision.** Model **B** — density porosity with each well's median core grain density as matrix (2.67 g/cc; 2.70 in F-O6)
and ρf = 1.0 — is nearly unbiased field-wide (−0.2 pu) with RMSE 2.25 pu. Only **F-O3** keeps a systematic −2.9 pu offset, so it
gets a +2.9 pu calibration (model C; its RMSE drops from 3.3 to 1.7 pu). The neutron–density gas formula (D) is worse and
correlates poorly in F-O3/F-S1 (neutron units and gas effect are uncertain), and subtracting a shale term (E) makes the
match worse — confirming core measures **total** porosity. → **PHIT = model B (+ F-O3 calibration).**

In [ ]:
# Cell 5 — blind test: predict each well with parameters from the other wells only (no per-well calibration)
K = cl[(cl.kkl_md > 0.001) & cl.phi_pct.notna()]
lowo = []
for w in sorted(ev.well.unique()):
    tr, te = ev[ev.well != w], ev[ev.well == w]; rg = tr.rhog.median()
    st = pm.error_stats(100 * (rg - te.RHOB_USE) / (rg - 1.0), te.phi_pct)
    f = pm.fit_logk(K[K.well != w].phi_pct, K[K.well != w].kkl_md); kt = K[K.well == w]
    phik = 100 * (rg - kt.RHOB_USE) / (rg - 1.0)
    sk = pm.error_stats(np.log10(pm.perm_from_phi(phik / 100, f["a"], f["b"])), np.log10(kt.kkl_md))
    lowo.append(dict(well=w, phi_n=st["n"], phi_bias_pu=st["bias"], phi_rmse_pu=st["rmse"],
                     logk_n=sk["n"], logk_bias_dex=sk["bias"], logk_rmse_dex=sk["rmse"]))
lowo = pd.DataFrame(lowo); lowo.to_csv(OUT_DIR / "blind_test_leave_one_well_out.csv", index=False); lowo

**Reading the blind test.** Porosity transfers to an unseen well with RMSE 2.1–2.6 pu (3.3 pu in F-O3, its density offset).
Permeability does **not** transfer: a field transform under-predicts F-O1 by ~1.2 decades (×17) — its telodiagenetic
enhanced-porosity sands are far more permeable at a given porosity than the background USM. One poro-perm line cannot
represent both, which is the quantitative case for diagenetic rock typing (Paper 1 / Paper 3).

## Step 3 — permeability

In [ ]:
# Cell 6 — transforms per well and field; porosity at 0.1 mD
fits = [dict(well="FIELD", **pm.fit_logk(K.phi_pct, K.kkl_md))]
for w, g in K.groupby("well"): fits.append(dict(well=w, **pm.fit_logk(g.phi_pct, g.kkl_md)))
fits = pd.DataFrame(fits); fits["phi_at_0.1mD"] = (np.log10(0.1) - fits.a) / fits.b
fits["frac_k_gt_1mD"] = fits.well.map((K.kkl_md > 1).groupby(K.well).mean()).round(2)
fits.round(3).to_csv(OUT_DIR / "poro_perm_transforms.csv", index=False); fits.round(3)

In [ ]:
# Cell 7 — NMR permeability vs core in F-O4 (the only cored well with NMR)
g = cl[(cl.well == "F-O4") & (cl.kkl_md > 0.001) & cl.KTIM.notna()]
nmr = []
for cv in ["KTIM", "KSDR"]:
    st = pm.error_stats(np.log10(g[cv].clip(lower=1e-4)), np.log10(g.kkl_md))
    nmr.append(dict(curve=cv, **st, median_core_over_nmr=round(float(np.median(g.kkl_md / g[cv].clip(lower=1e-4))), 2)))
nmr = pd.DataFrame(nmr); KTIM_FACTOR = float(nmr.loc[nmr.curve == "KTIM", "median_core_over_nmr"].iloc[0])
f4 = fits[fits.well == "F-O4"].iloc[0]
print(f"F-O4: poro-perm transform r = {f4.r:.2f}; NMR KTIM r(log) = {nmr.r.iloc[0]:.2f} -> use KTIM x {KTIM_FACTOR}")
nmr

**Permeability model for the curves.** Per-well transform where it is meaningful (|r| ≥ 0.55: F-O1, F-O2, F-O3, F-O6, F-S1);
**calibrated NMR** (KTIM × factor from F-O4 core) in F-O4 and F-O8 — in F-O4 NMR and the poro-perm transform predict core about
equally well (r ≈ 0.5 in log space), but NMR is independent of the density log and is the only option in uncored F-O8; field transform in F-R1 (no core, no NMR). The ~0.6 dex
scatter (×4) around every transform is the uncertainty to carry into Paper 2's Monte Carlo.

## Step 4 — curves and figures → composite v4

In [ ]:
# Cell 8 — PHIT, PERM
RHOMA = rhog.to_dict(); RHOMA_DEFAULT = float(rhog.median()); CALIB = {"F-O3": float(-bias_B["F-O3"])}
TF = {r.well: (r.a, r.b) for r in fits.itertuples()}
src = {}
for w in WELLS:
    d = logs[w]; rm = RHOMA.get(w, RHOMA_DEFAULT)
    d["PHIT"] = pm.phit_density(d.RHOB_USE, rm, 1.0, CALIB.get(w, 0.0)).mask(d.BADHOLE == 1)
    a, b = TF["FIELD"]; d["PERM_TF_FIELD"] = pm.perm_from_phi(d.PHIT, a, b)
    if w in ("F-O4", "F-O8") and "KTIM" in d:
        d["PERM"] = (d.KTIM * KTIM_FACTOR).clip(upper=2000); src[w] = f"NMR KTIM x {KTIM_FACTOR} (F-O4 core calibration)"
    elif w in TF and abs(fits.set_index("well").loc[w, "r"]) >= 0.55:
        d["PERM"] = pm.perm_from_phi(d.PHIT, *TF[w]); src[w] = f"well transform log10k = {TF[w][0]:.2f} + {TF[w][1]:.3f}*phi(%)"
    else:
        d["PERM"] = d.PERM_TF_FIELD; src[w] = "field transform"
    src[w] = f"rho_ma {rm:.3f}; calib {CALIB.get(w, 0):+.2f} pu; PERM: {src[w]}"
pd.Series(src, name="model")

In [ ]:
# Cell 9 — figures: core vs PHIT, and poro-perm by well
ev["PHIT_pu"] = ev.apply(lambda r: 100 * logs[r.well].PHIT.iloc[int(np.searchsorted(logs[r.well].index, r.depth_log_m))], axis=1)
cored = sorted(ev.well.unique())
f, ax = plt.subplots(2, 3, figsize=(11, 7), sharex=True, sharey=True)
for a, w in zip(ax.flat, cored):
    g = ev[ev.well == w]; a.scatter(g.phi_pct, g.PHIT_pu, s=8, alpha=0.6); a.plot([0, 20], [0, 20], color="0.4", lw=0.8)
    st = pm.error_stats(g.PHIT_pu, g.phi_pct); a.set_title(f"{w}  bias {st['bias']:+.1f}  RMSE {st['rmse']:.1f} pu", fontsize=9)
    a.set_xlim(0, 20); a.set_ylim(0, 20); a.grid(alpha=0.3)
for a in ax[-1]: a.set_xlabel("core porosity (%)")
for a in ax[:, 0]: a.set_ylabel("log PHIT (%)")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_core_vs_PHIT.png", dpi=150); plt.show()

f, a = plt.subplots(figsize=(6.5, 5))
for w, g in K.groupby("well"): a.scatter(g.phi_pct, g.kkl_md, s=7, alpha=0.6, label=w)
x = np.linspace(0, 18, 50); r = fits.set_index("well").loc["FIELD"]
a.plot(x, 10 ** (r.a + r.b * x), color="k", lw=1.2, label="field transform")
a.axhline(0.1, color="0.5", ls="--", lw=0.8); a.set_yscale("log"); a.set_xlabel("core porosity (%)"); a.set_ylabel("core permeability (mD)")
a.legend(fontsize=7, ncol=2); a.grid(alpha=0.3); plt.tight_layout(); plt.savefig(OUT_DIR / "fig_poro_perm.png", dpi=150); plt.show()

In [ ]:
# Cell 10 — write v4 (v3 curves + VSH_GR, VSH_GRL, VSH_TH, PHIT, PERM, PERM_TF_FIELD, RHOB_USE; POTA in %)
NEWU = dict(VSH_GR="V/V", VSH_GRL="V/V", VSH_TH="V/V", PHIT="V/V", PERM="mD", PERM_TF_FIELD="mD", RHOB_USE="g/cm3", POTA="%")
for w in WELLS:
    d = logs[w]
    if w == "F-O6": d = d.drop(columns=[c for c in ("THOR", "URAN") if c in d])     # keep original HTHO/HURA names
    units = {c.mnemonic: c.unit for c in lasio.read(str(V3_DIR / f"{w}_composite_v3.las")).curves}; units.update(NEWU)
    d.to_csv(OUT_DIR / f"{w}_composite_v4.csv", float_format="%.5f")
    las = lasio.LASFile(); las.well["WELL"].value = w; las.well["FLD"].value = "F-O Gas Field"
    las.other = f"{w} composite v4 (Notebook 03). {src[w]}. VSH endpoints in vsh_endpoints.csv. POTA in %."
    las.append_curve("DEPT", d.index.to_numpy(), unit="M")
    for c in d.columns: las.append_curve(c, d[c].to_numpy(), unit=units.get(c, ""))
    las.write(str(OUT_DIR / f"{w}_composite_v4.las"), version=2.0, fmt="%.5f")
summary = pd.DataFrame([dict(well=w, PHIT_window_median=round(float(logs[w].PHIT.loc[slice(*pm.WINDOWS[w])].median()), 3),
                             PERM_window_median_mD=round(float(logs[w].PERM.loc[slice(*pm.WINDOWS[w])].median()), 3),
                             VSH_window_median=round(float(logs[w].VSH_GR.loc[slice(*pm.WINDOWS[w])].median()), 2)) for w in WELLS])
summary.to_csv(OUT_DIR / "reservoir_summary_v4.csv", index=False)
summary   # medians over the whole reservoir window, sands and shales together (net-pay comes in Notebook 04)

## Methods text you can adapt
*Total porosity was computed from bulk density with a matrix density equal to the median core grain density of each well
(2.67 g/cm³; 2.70 g/cm³ in F-O6) and a fluid density of 1.0 g/cm³. Against 766 good-hole core plugs from six wells this model
is nearly unbiased (mean error −0.2 p.u.; RMSE 2.25 p.u.); F-O3 required a +2.9 p.u. calibration. In a leave-one-well-out test the
porosity model reproduced core in unseen wells with RMSE 2.1–2.6 p.u. In contrast, a single field-wide porosity–permeability
transform under-predicted the telodiagenetically enhanced sands of F-O1 by 1.2 decades, demonstrating that permeability is
controlled by diagenetic rock type rather than porosity alone.*

**Next (Notebook 04): saturation** — Rw, Archie m/n from the F-O2 and F-O6 SCAL, Waxman–Smits with Qv, against the LLD/RLA5
resistivities, then net-pay cut-offs.